# Notebook 02 — Data Cleaning
## Telecom Customer Churn and Retention Intelligence Dashboard

---

**Purpose:** Clean the raw telecom customer dataset based on findings from Phase 2 & Phase 3 data audits. Standardize naming, fix data types, handle missing values, and export the cleaned staging dataset.

**Input:** `data/raw/Telco-Customer-Churn.csv`

**Output:** `data/staging/telco_staging.csv`

**Key Cleaning Operations:**
1. Rename columns to consistent `snake_case`
2. Strip whitespace from all string/text columns
3. Remove duplicate rows & verify customer ID uniqueness
4. Fix `total_charges` data type (`object` -> `float64`)
5. Impute missing `total_charges` (set to `0` where `tenure == 0`)
6. Validate numeric ranges (no negative values)
7. Verify categorical consistency
8. Save staging dataset with schema verification

---

## Section 1 — Import Libraries and Setup

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 60)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Libraries imported successfully.')

## Section 2 — Load Raw Dataset

We load the raw CSV and make an explicit copy (`df = df_raw.copy()`) to preserve the raw dataset integrity.

In [ ]:
PROJECT_ROOT = Path.cwd().parent
RAW_FILE = PROJECT_ROOT / 'data' / 'raw' / 'Telco-Customer-Churn.csv'

df_raw = pd.read_csv(RAW_FILE)
df = df_raw.copy()

print(f'Raw data loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')

## Section 3 — Rename Columns to `snake_case`

**Why:** The raw dataset contains inconsistent column names (camelCase, PascalCase, lowercase like `gender`). Standardizing to `snake_case` makes querying in SQL and coding in Python and Power Query uniform and error-free.

In [ ]:
column_rename_map = {
    'customerID': 'customer_id',
    'gender': 'gender',
    'SeniorCitizen': 'senior_citizen',
    'Partner': 'partner',
    'Dependents': 'dependents',
    'tenure': 'tenure',
    'PhoneService': 'phone_service',
    'MultipleLines': 'multiple_lines',
    'InternetService': 'internet_service',
    'OnlineSecurity': 'online_security',
    'OnlineBackup': 'online_backup',
    'DeviceProtection': 'device_protection',
    'TechSupport': 'tech_support',
    'StreamingTV': 'streaming_tv',
    'StreamingMovies': 'streaming_movies',
    'Contract': 'contract',
    'PaperlessBilling': 'paperless_billing',
    'PaymentMethod': 'payment_method',
    'MonthlyCharges': 'monthly_charges',
    'TotalCharges': 'total_charges',
    'Churn': 'churn'
}

df = df.rename(columns=column_rename_map)
print('Columns renamed successfully:')
print(list(df.columns))

## Section 4 — Strip Whitespace from Text Columns

**Why:** Leading or trailing whitespace in categorical strings (e.g. `' Month-to-month'` or `'DSL '`) causes silent mismatch errors in filters, SQL queries, and DAX relationships.

In [ ]:
object_cols = df.select_dtypes(include='object').columns
for col in object_cols:
    df[col] = df[col].astype(str).str.strip()

print(f'Stripped leading/trailing whitespace across {len(object_cols)} object columns.')

## Section 5 — Deduplication & Primary Key Verification

**Why:** We must ensure each row represents one distinct customer and that `customer_id` is strictly unique.

In [ ]:
# Check for exact duplicate rows
exact_dups = df.duplicated().sum()
print(f'Exact duplicate rows: {exact_dups}')
if exact_dups > 0:
    df = df.drop_duplicates().copy()
    print(f'Dropped {exact_dups} duplicate rows.')

# Check for customer_id uniqueness
id_dups = df['customer_id'].duplicated().sum()
print(f'Duplicate customer IDs: {id_dups}')
if id_dups > 0:
    df = df.drop_duplicates(subset=['customer_id'], keep='first').copy()
    print(f'Retained first occurrence for duplicate customer IDs.')

assert df['customer_id'].nunique() == len(df), 'Customer ID must be unique across all rows!'

## Section 6 — Convert & Impute `total_charges`

**Why:** In the raw dataset, `total_charges` is loaded as an `object` because 11 rows have blank spaces (`' '`). All 11 of these rows correspond to customers with `tenure == 0` (brand-new accounts that have not completed their first billing cycle). Converting to numeric changes `' '` to `NaN`, which we then impute with `0.0`.

In [ ]:
# Convert to numeric (blanks become NaN)
df['total_charges'] = pd.to_numeric(df['total_charges'], errors='coerce')

missing_tc = df['total_charges'].isna().sum()
print(f'NaN total_charges count after numeric coercion: {missing_tc}')

# Impute 0.0 for zero-tenure customers
zero_tenure_mask = df['total_charges'].isna() & df['tenure'].eq(0)
print(f'NaN total_charges with tenure == 0: {zero_tenure_mask.sum()}')
df.loc[zero_tenure_mask, 'total_charges'] = 0.0

# Check if any unexpected NaNs remain
remaining_nans = df['total_charges'].isna().sum()
print(f'Remaining NaN total_charges: {remaining_nans}')
assert remaining_nans == 0, 'All total_charges values must be populated!'

## Section 7 — Validate Numeric Ranges & Data Integrity

**Why:** Business logic assertions ensure that no invalid data passes to downstream models or dashboards (e.g. tenure cannot be negative, monthly charges must be positive).

In [ ]:
# Assertions for data validation
assert (df['tenure'] >= 0).all(), 'Tenure cannot be negative'
assert (df['tenure'] <= 120).all(), 'Tenure exceeds expected upper bound (10 years)'
assert (df['monthly_charges'] > 0).all(), 'Monthly charges must be strictly positive'
assert (df['total_charges'] >= 0).all(), 'Total charges cannot be negative'

print('All numeric range assertions passed successfully:')
display(df[['tenure', 'monthly_charges', 'total_charges']].describe())

## Section 8 — Verify Categorical Consistency

**Why:** Ensure all category values match expected domain definitions.

In [ ]:
expected_categories = {
    'gender': {'Female', 'Male'},
    'senior_citizen': {0, 1},
    'partner': {'Yes', 'No'},
    'dependents': {'Yes', 'No'},
    'phone_service': {'Yes', 'No'},
    'multiple_lines': {'No', 'Yes', 'No phone service'},
    'internet_service': {'DSL', 'Fiber optic', 'No'},
    'contract': {'Month-to-month', 'One year', 'Two year'},
    'paperless_billing': {'Yes', 'No'},
    'churn': {'Yes', 'No'}
}

for col, expected_vals in expected_categories.items():
    actual_vals = set(df[col].unique())
    assert actual_vals.issubset(expected_vals), f'Unexpected values found in {col}: {actual_vals - expected_vals}'
    print(f'✓ {col:18s}: {sorted(list(actual_vals))}')

print('\nAll categorical validations passed.')

## Section 9 — Save Cleaned Staging Dataset

**Why:** We write the cleaned dataset to `data/staging/telco_staging.csv`. This provides an audited checkpoint before feature engineering in Phase 5.

In [ ]:
staging_dir = PROJECT_ROOT / 'data' / 'staging'
staging_dir.mkdir(parents=True, exist_ok=True)

staging_file = staging_dir / 'telco_staging.csv'
df.to_csv(staging_file, index=False)

print(f'Cleaned staging dataset saved successfully to:')
print(f'  {staging_file}')
print(f'Final Staging Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')

## Section 10 — Before vs After Cleaning Audit Summary

In [ ]:
audit_comparison = pd.DataFrame({
    'Metric': [
        'Total Rows',
        'Total Columns',
        'TotalCharges dtype',
        'TotalCharges Blank Count',
        'TotalCharges Null Count',
        'Exact Duplicate Rows',
        'Unique Customer IDs'
    ],
    'Raw Dataset (Before)': [
        df_raw.shape[0],
        df_raw.shape[1],
        str(df_raw['TotalCharges'].dtype),
        (df_raw['TotalCharges'].astype(str).str.strip() == '').sum(),
        df_raw['TotalCharges'].isna().sum(),
        df_raw.duplicated().sum(),
        df_raw['customerID'].nunique()
    ],
    'Staging Dataset (After)': [
        df.shape[0],
        df.shape[1],
        str(df['total_charges'].dtype),
        0,
        df['total_charges'].isna().sum(),
        df.duplicated().sum(),
        df['customer_id'].nunique()
    ]
})

display(audit_comparison)

---

## Next Step

Proceed to **`03_feature_engineering.ipynb`** (Phase 5) where we:
1. Create tenure cohorts (`0-12m`, `13-24m`, `25-48m`, `49-72m`)
2. Create monthly charges tiers
3. Engineer service adoption flags & service count
4. Create customer value flags and high-risk indicators

---
**Data Cleaning Phase Complete.**